## Libraries

In [2]:
# ---------------------------------------
# 1) Build fake “simtk” modules
import types, sys

# Create a new empty module object to stand in for “simtk”
simtk = types.ModuleType("simtk")
sys.modules["simtk"] = simtk

# Create sub‐modules for simtk.openmm, simtk.openmm.app, and simtk.unit
simtk_openmm     = types.ModuleType("simtk.openmm")
simtk_openmm_app = types.ModuleType("simtk.openmm.app")
simtk_unit       = types.ModuleType("simtk.unit")

# Register them in sys.modules so "import simtk.openmm..." works
sys.modules["simtk.openmm"]     = simtk_openmm
sys.modules["simtk.openmm.app"] = simtk_openmm_app
sys.modules["simtk.unit"]       = simtk_unit

# Attach the sub‐modules as attributes on the “simtk” module
simtk.openmm     = simtk_openmm
simtk.openmm.app = simtk_openmm_app
simtk.unit       = simtk_unit

# ---------------------------------------
# 2) Copy attributes from real openmm into our fake simtk.* modules
import openmm
import openmm.app as _app
import openmm.unit as _unit

# Copy everything defined in the top‐level openmm into simtk.openmm
for name in dir(openmm):
    setattr(simtk_openmm, name, getattr(openmm, name))

# Copy every attribute of openmm.app into simtk.openmm.app
for name in dir(_app):
    setattr(simtk_openmm_app, name, getattr(_app, name))

# Copy every attribute of openmm.unit into simtk.unit
for name in dir(_unit):
    setattr(simtk_unit, name, getattr(_unit, name))
# ---------------------------------------


import mbuild as mb
from mBuild_defect import CLP_box
from clp_builder_defect import CLP_helix as build_clp
import numpy as np
if not hasattr(np, 'float'):
    np.float = float

from foyer import Forcefield    # this will now find "simtk.openmm.app.element"" and "import mbuild as mb
import numpy as np
from foyer import Forcefield
import parmed as pmd
from simtk import unit, openmm
from simtk.openmm import app

## Building the model

In [3]:
def random_rotation_matrix():
    u1, u2, u3 = np.random.uniform(size=3)
    q1 = np.sqrt(1 - u1) * np.sin(2 * np.pi * u2)
    q2 = np.sqrt(1 - u1) * np.cos(2 * np.pi * u2)
    q3 = np.sqrt(u1) * np.sin(2 * np.pi * u3)
    q4 = np.sqrt(u1) * np.cos(2 * np.pi * u3)
    return np.array([
        [1 - 2 * (q3**2 + q4**2), 2 * (q2*q3 - q1*q4), 2 * (q2*q4 + q1*q3)],
        [2 * (q2*q3 + q1*q4), 1 - 2 * (q2**2 + q4**2), 2 * (q3*q4 - q1*q2)],
        [2 * (q2*q4 - q1*q3), 2 * (q3*q4 + q1*q2), 1 - 2 * (q2**2 + q3**2)]
    ])

def apply_rotation(compound, rot_matrix):
    coords = compound.xyz
    center = compound.center
    coords -= center
    rotated = np.dot(coords, rot_matrix.T)
    compound.xyz = rotated + center

def has_overlap(new_pos, prev_pos_list, min_dist=1.5):
    for prev in prev_pos_list:
        dists = np.linalg.norm(new_pos[:, None, :] - prev[None, :, :], axis=-1)
        if np.any(dists < min_dist):
            return True
    return False

# Create the custom sequence: (POG)5-(POW)-(POG)6
sequence = "POG" * 5 + "POW" + "POG" * 6

# Parameters
num_copies = 1  # how many trimer units you want to randomly place

combined = mb.Compound()
prev_positions = []

for i in range(num_copies):
    placed = False
    attempts = 0
    while not placed and attempts < 100:
        helix = build_clp(sequence)  # This should build a triple helix from the custom sequence
        rot = random_rotation_matrix()
        apply_rotation(helix, rot)

        shift = np.random.uniform(-20, 20, size=3)
        helix.translate(shift)

        coords = helix.xyz
        if not has_overlap(coords, prev_positions):
            combined.add(helix)
            prev_positions.append(coords)
            placed = True
        attempts += 1

combined.save("POG5POWPOG6_1trimer.pdb", overwrite=True)


/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<BBP pos=(-9.4894, 9.8403, 21.4657), 0 bonds, id: 23454654916880>" is element: "B"
  warn(
/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<HBP pos=(-9.7024, 10.0201, 21.7091), 0 bonds, id: 23454654884368>" is element: "H"
  warn(
/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<BBO pos=(-9.8972, 9.6415, 21.2556), 0 bonds, id: 23454654827984>" is element: "B"
  warn(
/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<BBG pos=(-10.3050, 9.4427, 21.0455), 0 bonds, id: 23454646551504>" is element: "B"
  warn(
/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<HBG pos=(-10.5180, 9.6224, 21.2888), 0 bonds, id: 23455418369616>" is ele

In [9]:
def random_rotation_matrix():
    u1, u2, u3 = np.random.uniform(size=3)
    q1 = np.sqrt(1 - u1) * np.sin(2 * np.pi * u2)
    q2 = np.sqrt(1 - u1) * np.cos(2 * np.pi * u2)
    q3 = np.sqrt(u1) * np.sin(2 * np.pi * u3)
    q4 = np.sqrt(u1) * np.cos(2 * np.pi * u3)
    return np.array([
        [1 - 2 * (q3**2 + q4**2), 2 * (q2*q3 - q1*q4), 2 * (q2*q4 + q1*q3)],
        [2 * (q2*q3 + q1*q4), 1 - 2 * (q2**2 + q4**2), 2 * (q3*q4 - q1*q2)],
        [2 * (q2*q4 - q1*q3), 2 * (q3*q4 + q1*q2), 1 - 2 * (q2**2 + q3**2)]
    ])

def apply_rotation(compound, rot_matrix):
    coords = compound.xyz
    center = compound.center
    coords -= center
    rotated = np.dot(coords, rot_matrix.T)
    compound.xyz = rotated + center

def has_overlap(new_pos, prev_pos_list, min_dist=1.5):
    for prev in prev_pos_list:
        dists = np.linalg.norm(new_pos[:, None, :] - prev[None, :, :], axis=-1)
        if np.any(dists < min_dist):
            return True
    return False

# Parameters
num_repeats = 12
num_copies = 1
sequence = "POG" * num_repeats

combined = mb.Compound()
prev_positions = []

for i in range(num_copies):
    placed = False
    attempts = 0
    while not placed and attempts < 100:
        helix = build_clp(sequence)
        rot = random_rotation_matrix()
        apply_rotation(helix, rot)

        shift = np.random.uniform(-20, 20, size=3)
        helix.translate(shift)

        coords = helix.xyz
        if not has_overlap(coords, prev_positions):
            combined.add(helix)
            prev_positions.append(coords)
            placed = True
        attempts += 1

combined.save("POG12_1trimer.pdb", overwrite=True)


/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<BBP pos=(-4.2598, 16.7631,-6.2489), 0 bonds, id: 23454649298768>" is element: "B"
  warn(
/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<HBP pos=(-4.4505, 16.5945,-5.9804), 0 bonds, id: 23454648219664>" is element: "H"
  warn(
/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<BBO pos=(-4.0840, 16.3208,-6.4019), 0 bonds, id: 23454649628432>" is element: "B"
  warn(
/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<BBG pos=(-3.9081, 15.8785,-6.5549), 0 bonds, id: 23454648136528>" is element: "B"
  warn(
/mnt/home/rsantos/openmm-env/lib/python3.11/site-packages/mbuild/conversion.py:970: UserWarning: Guessing that "<HBG pos=(-4.0988, 15.7098,-6.2864), 0 bonds, id: 23454650170768>" is elemen

In [10]:
# Load the original PDB with _BBP, _HBP etc.
premd = pmd.load_file("POG12_1trimer.pdb", structure=True)

#Set atom.type based on bead name
valid_types = {"BBP", "BBO", "BBG", "BBK", "BBD", "HBP", "HBG", "INC", "INA"}
for atom in premd.atoms:
    bead_type = atom.name.lstrip('_')
    if bead_type not in valid_types:
        raise KeyError(f"Bad bead_type '{atom.name}' → '{bead_type}'")

In [6]:
import argparse
import itertools
import string
from sys import stdout
from simtk import openmm, unit
from openmm import app
from openmm import app as appmod
from openmm import XmlSerializer
from openmm.app import Topology
import numpy as np

# =============================================================================
# Parse command-line arguments
# =============================================================================
# Manually set your input variables
pdb_file = "POG5POWPOG6_1trimervs2.pdb"
outprefix = "POG5POWPOG6_1trimervs3"


# =============================================================================
# Load structure and build topology
# =============================================================================
pdb = app.PDBFile(pdb_file)

top = Topology()
top.setPeriodicBoxVectors(pdb.topology.getPeriodicBoxVectors())

'''
def generate_chain_ids():
    letters = string.ascii_uppercase + string.ascii_lowercase
    for size in itertools.count(1):
        for combo in itertools.product(letters, repeat=size):
            yield ''.join(combo)
'''
def generate_chain_ids():
    letters = list(string.ascii_uppercase) + list(string.ascii_lowercase)
    for ch in letters:
        yield ch
    for combo in itertools.product(letters, repeat=2):
        yield ''.join(combo)
        
chain_ids = generate_chain_ids()
atom_map = {}
new_residues = []
atoms_per_chain = 60
current_chain = top.addChain(id=next(chain_ids))
atom_counter = 0

for res in pdb.topology.residues():
    if atom_counter >= atoms_per_chain:
        current_chain = top.addChain(id=next(chain_ids))
        atom_counter = 0
    new_res = top.addResidue(res.name, current_chain)
    atoms = []
    for atom in res.atoms():
        new_atom = top.addAtom(atom.name, atom.element, new_res)
        atom_map[atom] = new_atom
        atoms.append(new_atom)
        atom_counter += 1
    new_residues.append((new_res, atoms))

for bond in pdb.topology.bonds():
    atom1 = atom_map[bond[0]]
    atom2 = atom_map[bond[1]]
    if atom1.residue.chain == atom2.residue.chain:
        top.addBond(atom1, atom2)

for chain in top.chains():
    residues = list(chain.residues())
    for i in range(len(residues) - 1):
        atoms1 = {atom.name: atom for atom in residues[i].atoms()}
        atoms2 = {atom.name: atom for atom in residues[i + 1].atoms()}
        if 'BBG' in atoms1 and 'BBP' in atoms2:
            top.addBond(atoms1['BBG'], atoms2['BBP'])
        if 'BBW' in atoms1 and 'BBP' in atoms2:
            top.addBond(atoms1['BBW'], atoms2['BBP'])

for res, atoms in new_residues:
    atom_dict = {atom.name: atom for atom in atoms}
    
    # Standard (POG) residue types
    if 'BBP' in atom_dict and 'HBP' in atom_dict:
        top.addBond(atom_dict['BBP'], atom_dict['HBP'])
    if 'BBP' in atom_dict and 'BBO' in atom_dict:
        top.addBond(atom_dict['BBP'], atom_dict['BBO'])
    if 'BBO' in atom_dict and 'BBG' in atom_dict:
        top.addBond(atom_dict['BBO'], atom_dict['BBG'])
    if 'BBG' in atom_dict and 'HBG' in atom_dict:
        top.addBond(atom_dict['BBG'], atom_dict['HBG'])

    # POW residue with W-type backbone and H-bond
    if 'BBO' in atom_dict and 'BBW' in atom_dict:
        top.addBond(atom_dict['BBO'], atom_dict['BBW'])
    if 'BBW' in atom_dict and 'HBW' in atom_dict:
        top.addBond(atom_dict['BBW'], atom_dict['HBW'])

modeller = app.Modeller(top, pdb.positions)
with open(f"{outprefix}.pdb", "w") as f:
    app.PDBFile.writeFile(modeller.topology, modeller.positions, f)
